# Final Project - Basics of Mobile Robotics
### Prof. Francesco Mondada
###  EPFL - Fall 2024
###  Group 04
- Jules BERVILLE
- Jin KILIDJIAN
- Corentin PLUMET
- Charles PROFFIT

## Table of Contents
- [Introduction](#introduction)
- [Global environment](#global-environment)
- [Libraries and code structure](#libraries-and-code-structure)
- [Vision](#vision)
- [Path planning](#path-planning)
  - [Obstacle size increase (padding)](#obstacle-size-increase-padding)
  - [Algorithm choice](#algorithm-choice)
  - [Simulation](#simulation)
  - [Path Sampling](#path-sampling)
- [Filtering](#filtering)
  - [Q matrix - theoretical values](#q-matrix---theoretical-values)
  - [R matrix - theoretical values](#r-matrix---theoretical-values)
  - [Tuning](#tuning)
- [Global navigation](#global-navigation)
- [Local avoidance](#local-avoidance)
  - [Obstacle avoidance](#obstacle-avoidance)
  - [Kidnapping scenarios](#kidnapping-scenarios)
- [Live plots](#live-plots)
- [Video](#video)
- [Conclusion](#conclusion)


## Introduction
This report will outline the different choices we made to achieve the goals set for this project. 
The project consists of controlling a Thymio using an external camera and all of its internal sensors to navigate around a map, avoid local and global obstacles and find a path to a goal.
This Jupyter Notebook will go through the different independant modules of the code and their technical aspects along the way. The report section doesn't have running code. At the end of this notebook you will find the code to run the project. This project is strongly based on the guidelines and theory provided by the course MICRO-452 Basics of Mobile Robotics of Prof. Francesco Mondada.

## Global environment
We chose to use a white A0 paper sheet as the environment. It has 4 different ArUco on each of its corners to allow for the camera to map every thing. We also added black 2D shapes to represent obstacles. Finally we chose to also use ArUco markers to represent the goal and the Thymio.
Here is the map that we chose to use for this project:

<img src="./Images/map.png"  style="display: block; margin-left: auto; margin-right: auto; width: 50%; height: auto;">

The ArUco markers for the goal and the Thymio are the following:
<div style="display: flex; justify-content: center;">
    <img src="./Images/goal.png"  style="width: 20%; height: auto; margin-right: 10px;">
    <img src="./Images/thymio.png"  style="width: 20%; height: auto;">
</div>


ArUco were chosen for their reliability and ease of use. It provides a position as wall as an orientation which is ideal for the Thymio positioning for example. It has good documentation and ready-to-use libraries in Python. We used the `cv2.aruco` library to detect them.

## Libraries and code structure
We chose to implement the different modules in different .py files to make the code more readable. We also added a parameters.py file to store all the parameters of the project which allowed us to debug more easily and tune the control faster as it is quite sensible to environmental conditions (lighting mainly).

The program execution is articulated around two parts : at first the initialization of the environment, using the camera readings, and then a loop containing the main behavior of the robot. The main loop can be divided in 4 parts, easily identifiable:
- the readings of the camera values (additionnally a small finite machine to determine the mode of the camera, either NOMINAL or OBSTRUCTED)
- the readings ot Thymio sensor values
- the estimation of the state of the robot (x,y,angle,linear and angular speeds)
- the finite state machine deciding the behavior of the robot and its mode (FOLLOW_PATH, OBSTACLE_AVOIDANCE, KIDNAPPED)

<figure>
<img src="./Images/FSM.png" alt="Finite state machine of the Thymio" style="display: block; margin-left: auto; margin-right: auto; width: 50%; height: auto;">
<figcaption style="text-align: center;">Finite state machine of the Thymio </figcaption>
</figure>

## Vision 
Initially, we aimed to use color differentiation in the HSV color space for the vision system, assigning different colors to the obstacles, goal, and robot. However, we encountered two major challenges with this approach: the system's dependence on environmental conditions and the need for precise camera angulation to avoid image distortion. Because the HSV-based detection relied heavily on consistent color thresholds, even slight changes in lighting or surroundings required constant recalibration. Additionally, imperfect camera angles caused distortions that were difficult to correct. These issues led us to switch to using ArUco markers and maintain black obstacles against a white background. We took inspiration from the following sources: [Geeks for geeks](https://www.geeksforgeeks.org/detecting-aruco-markers-with-opencv-and-python-1/), [Pyimage search](https://pyimagesearch.com/2020/12/21/detecting-aruco-markers-with-opencv-and-python/) and [OpenCV](https://docs.opencv.org/4.x/d5/dae/tutorial_aruco_detection.html).

We used a total of 6 ArUco markers: 4 to mark the corners of the map (ID: 0, 1, 2, 3), 1 for the robot (ID: 4), and 1 for the goal (ID: 5). The 4 corner markers are crucial because, beyond defining the map's boundaries, they allow us to correct image distortions by applying a transformation matrix to change the perspective of the video feed. By using the outermost corner of each marker, we define a coordinate system on a (297, 210) grid, with each cell representing an area of 16 square millimeters on our A0-sized map. We kept this map dimension because it provided sufficient precision while minimizing the load for path planning, allowing us to work with a smaller, more manageable map.

We decided to apply a Gaussian blur in order to reduce the noise, most importantly on the edges of our map where we noticed a lot of it. For that we decided to use a kernel size of 5x5 and a sigma of 1,25. These parameters were chosen in order to reduce the noise while retaining the natural sharpness of edges. We use the following Gaussian function:
$$G(x, y) = \frac{1}{2\pi\sigma^2} e^{-\frac{x^2 + y^2}{2\sigma^2}}$$
From that, by normalizing the results, we obtain the following Gaussian kernel:
$$\begin{bmatrix}
0.00854167 && 0.02230825 && 0.03072132 && 0.02230825 && 0.00854167 \\
0.02230825 && 0.05826239 && 0.08023475 && 0.05826239 && 0.02230825 \\
0.03072132 && 0.08023475 && 0.11049351 && 0.08023475 && 0.03072132 \\
0.02230825 && 0.05826239 && 0.08023475 && 0.05826239 && 0.02230825 \\
0.00854167 && 0.02230825 && 0.03072132 && 0.02230825 && 0.00854167 \\
\end{bmatrix}$$
The Aruco and obstacle detection processes were carried out after applying a Gaussian blur. We also adjusted the parameters of the detectMarkers() function to improve robustness against environmental changes and enhance edge precision, which is critical as corners play a significant role in subsequent functions. During the initialization of the map, we made sure that all Aruco were detected qnd if not display the frame with all the detected markers in order to identify the missing ones. Here is an example, with the Aruco ID3 not being detected.

<figure>
<img src="./Images/aruco_not_detected.png" alt="Waiting until good detection of aruco markers" style="display: block; margin-left: auto; margin-right: auto; width: 50%; height: auto;">
<figcaption style="text-align: center;">Waiting until good detection of aruco markers </figcaption>
</figure>

For obstacle detection, we kept everything black and white to simplify thresholding. Since the map is already in black and white, we first converted it to grayscale to streamline image processing. A threshold was then applied to create a binary image, but due to variations in the environment, we found it necessary to dynamically adjust this threshold. To address this, we implemented a loop that adjusts the threshold value as needed. Here is an example of this adjustement:


<figure style="display: flex; flex-direction: column; align-items: center;">
    <div style="display: flex; justify-content: center; gap: 10px;">
        <img src="./Images/threshold_before.png" alt="Threshold Before" style="width: 25%; height: auto;">
        <img src="./Images/threshold_after.png" alt="Threshold After" style="width: 25%; height: auto;">
    </div>
    <figcaption style="text-align: center; margin-top: 10px;">Before and after adjusting the threshold</figcaption>
</figure>

Using the resulting binary image, we identified all contours, for that we used the findContours() function. We chose the RETR_EXTERNAL for the contour retrieval mode to have the outermost countours to identify the boundaries of object. For the countour approximation method, we used CHAIN_APPROX_SIMPLE because our obstacles are simple mathematical objects. However, since the markers were also being detected as obstacles, we ensured the obstacles appeared larger than the markers, allowing us to sort the contours by size and select only the relevant ones. Specifically, we retained the four largest contours, corresponding to the four obstacles we intended to detect. This process provided us with the locations of all obstacles on the map.

## Path planning

### Obstacle size increase (padding)
To enhance safety during navigation, we increased the size of obstacles in the grid received from the vision system. This approach provides a margin of error, ensuring the robot avoids collisions with obstacles. Additionally, it simplifies navigation by allowing us to track only the Thymio's center without modeling its full dimensions.

#### Padding Depth Calculation
The padding depth is calculated using the following formula:

python	
  depth = int(params.PADDING_RATI0*(params.AXLE_LENGTH/2)/params.SIZE_CASE)


This formula determines the required padding around obstacles based on the security factor PADDING_RATIO. By adjusting this ratio, we can control the safety buffer around each obstacle.

#### Edge Padding for Map Boundaries
To ensure the robot remains within the map boundaries, we added a thin forbidden layer around the edges of the map, defined by the parameter PADDING_SIDE.

#### Visual Representation
Below is an example of a padded map that includes both obstacle padding and edge boundaries for improved navigation safety.

<div style="text-align: center;"> <img src="Images/a_star_vs_dijkstra/padded_map.png" alt="Padded Map Example" width="500" /> </div>

This padding strategy not only enhances safety but also simplifies the planning process by abstracting the robot's physical dimensions into the grid.

### Algorithm choice
For path planning mutiple options were available. First We had to choose between a "Road-map" or a "Cell decomposition" approach. We had already worked in the past with the a map following the "Cell decompostion" approach (within the frame of the Aerial Robotics course project) so we chose that kind of environment representation.

Once we chose to use a "Cell decomposition" approach, we had to choose between different algorithm to find a path between the starting cell and the goal cell. The 2 algorithm we considered were the A* algorithm and the Djikstra algorithm. While the A* is supposed to be faster to compute, with the Djikstra algorithm we have the assurance to find the shortest path.

For the A* algorithm we took the functions from the following source : [A* algorithm in python](https://www.geeksforgeeks.org/a-search-algorithm-in-python/).
<br>
For the Dijkstra algorithm the A* algorithm from the same source : [A* algorithm in python](https://www.geeksforgeeks.org/a-search-algorithm-in-python/).

We ran the to algorithm on the 3 different path to compute and got the following results:

#### Test 1: Start (x=20, y=50, angle=π/2), Goal (30, 270)
<div style="text-align: center;">
  <table>
    <tr>
      <td><strong>A Star</strong></td>
      <td><strong>Dijkstra</strong></td>
    </tr>
    <tr>
      <td><img src="Images/a_star_vs_dijkstra/a_star1.png" alt="A Star Test 1" width="500" /></td>
      <td><img src="Images/a_star_vs_dijkstra/dijkstra1.png" alt="Dijkstra Test 1" width="500" /></td>
    </tr>
    <tr>
      <td>Execution Time: 0.312927 seconds</td>
      <td>Execution Time: 0.353220 seconds</td>
    </tr>
  </table>
</div>

#### Test 2: Start (x=190, y=50, angle=3π/4), Goal (30, 270)
<div style="text-align: center;">
  <table>
    <tr>
      <td><strong>A Star</strong></td>
      <td><strong>Dijkstra</strong></td>
    </tr>
    <tr>
      <td><img src="Images/a_star_vs_dijkstra/a_star2.png" alt="A Star Test 2" width="500" /></td>
      <td><img src="Images/a_star_vs_dijkstra/dijkstra2.png" alt="Dijkstra Test 2" width="500" /></td>
    </tr>
    <tr>
      <td>Execution Time: 0.249407 seconds</td>
      <td>Execution Time: 0.572213 seconds</td>
    </tr>
  </table>
</div>

#### Test 3: Start (x=190, y=50, angle=π/2), Goal (190, 270)
<div style="text-align: center;">
  <table>
    <tr>
      <td><strong>A Star</strong></td>
      <td><strong>Dijkstra</strong></td>
    </tr>
    <tr>
      <td><img src="Images/a_star_vs_dijkstra/a_star3.png" alt="A Star Test 3" width="500" /></td>
      <td><img src="Images/a_star_vs_dijkstra/dijkstra3.png" alt="Dijkstra Test 3" width="500" /></td>
    </tr>
    <tr>
      <td>Execution Time: 0.153406 seconds</td>
      <td>Execution Time: 0.317986 seconds</td>
    </tr>
  </table>
</div>

We can see that for a very similar result, the A* was always faster and 2 times out of 3 it was 2 times faster so chose to stick with a A* algorithm.

### Simulation

In order to be able to test our parameters and algorithm without the robot we implemented a theoritical simulation of the robot trajectory in the map. The simulation is carried by the simu function, which use an object of the class Robot. The class has 2 main methods that are control_to_target which is the control algorithm that we implemented (Astolfi) and update which computes the linear and angular velocity of the robot to predict is next position at the next time step DT.


### Path Sampling

The problem we encountered when associating our control algorithm with the path computed by the A* is that the points to follow were to close to each other, causing an oscillation of the robot. Indeed, having very small distances between goals created large angular errors and therefore large motor response which was not ideal. The oscillations on the plot below are not very 'agressive' (low frequency and amplitude) because of a large error allowed when reaching the point (the DIST_ERROR constante):
<div style="text-align: center;">
<img src="Images/sampling/not_sampled1.png" alt="Path not sampled 1" width="350" />
<img src="Images/sampling/not_sampled2.png" alt="Path not sampled 2" width="350" />
<img src="Images/sampling/not_sampled3.png" alt="Path not sampled 3" width="350" />
</div>

To solve this problem we chose to sample down the path computed by the A*. This way the robot has less restriction on his path and it can be smoother. But the drawback is that the path to follow is now an approximation of the computed working path. Thus, it can happen that this new sampled trajectory crosses the padding (obstacle increased size). As the padding around the obstaclce is larger than half the robot's width, the robot doesn't run on real obstacle as long as the error in the new sampled path is not to large.

<div style="text-align: center;">
<img src="Images/sampling/sampling_20_1.png" alt="Path sampled every 20 points 1" width="350" />
<img src="Images/sampling/sampling_20_2.png" alt="Path sampled every 20 points 2" width="350" />
<img src="Images/sampling/sampling_20_3.png" alt="Path sampled every 20 points 3" width="350" />
</div>

As we can see above, with 1 point sampled out of 20, the trajectory is smoother than before with a small error (goes a little bit through the padding).


## Filtering 
The system at hand is nonlinear due to the cos and sin expressions. This restricts our choice of filtering algorithm to the Extended Kalman Filter (EKF)  and the Unscented Kalman Filter (UKF). Even though the UKF is more robust to nonlinearity, we chose to use the EKF because of its simplicity and its broad documentation. In fact the EKF is the same as the Kalman Filter seen in class but instead of using the A and B matrices, we use their Jacobian matrices to linearize the system at each step.
The second choice we had to make was to choose the states and inputs. We had the 2 following configurations :

1.  $ x = \begin{bmatrix} x_{t} \\ y_{t} \\ \theta_{t} \\ v \\ w \end{bmatrix}$

2. $ x = \begin{bmatrix} x_{t} \\ y_{t} \\ \theta_{t} \end{bmatrix}$
    $ u = \begin{bmatrix} v \\ w \end{bmatrix}$


In the end the 5 states version was selected because it was more precise when the camera was disabled. Having the linear and angular speeds of the Thymio allowed to have a good prediction of the next state and have a smoother speed measurements.

For the implementation of the EKF we took inspiration from the following papers : [EKF with python code example](https://automaticaddison.com/extended-kalman-filter-ekf-with-python-code-example/) and [Kalman and Bayesian Filters in Python](https://github.com/rlabbe/Kalman-and-Bayesian-Filters-in-Python/blob/master/11-Extended-Kalman-Filters.ipynb). 
The main steps are already described and commented in the code of the ekf() function : 

```python
    def ekf(dt, x_prev, P_prev, z, R):
      F= calculate_jacobian(x_prev, dt)

      # Prediction step 
      A=get_matrix_A(x_prev, dt)
      x = A @ x_prev

      # State covariance matrix prediction
      P = F @ P_prev @ F.T + Q

      # Innovation step
      y = z - H @ x

      # Innovation covariance
      S = H @ P @ H.T + R

      # Kalman gain
      K = P @ H.T @ np.linalg.inv(S)

      # Update step of state estimate
      x = x + K @ y

      # Update state covariance matrix
      P = P - K @ H @ P
    return x, P
```
The theory behind is quite well known so we will only discuss the part specific to our system.

For the prediction step, we need to compute the process matrix and its Jacobian matrix. This was done once analytically and is then called every time to be computed. 
We have the matrix A as follows : 

$$ A = \begin{bmatrix} 1 && 0 && 0 && \Delta t cos(\theta) && 0 \\ 
                        0 && 1 && 0 && \Delta t sin(\theta) && 0 \\
                        0 && 0 && 1 && 0 && \Delta t \\
                        0 && 0 && 0 && 1 && 0 \\
                        0 && 0 && 0 && 0 && 1 \end{bmatrix} $$

We can then deduce the Jacobian matrix as follows :
$$ F = \begin{bmatrix} 1 && 0 && -v \Delta t sin(\theta) && \Delta t cos(\theta) && 0 \\ 
                        0 && 1 && v \Delta t  cos(\theta) && \Delta t sin(\theta)&& 0 \\
                        0 && 0 && 1 && 0 && \Delta t \\
                        0 && 0 && 0 && 1 && 0 \\
                        0 && 0 && 0 && 0 && 1 \end{bmatrix} $$

The main challenge for the Kalman filter is to choose the right values for the R and Q matrices as it is very case dependant. 
### Q matrix - theoretical values
To find the Q matrix, we only relied on the speed states to derive all the variances. In fact in the process the position is integrated from the speed on the time elapsed.
For the speed process variance we calculated the variance of the speed of the Thymio by running it for 10s and measuring the distance it covers. We found the following results : 
$$ Q_v = 0.43 mm^2/s^2 \\
Q_w = 0.024 rad^2/s^2$$   


We then calculated the variance of the position and angle integrated from the speeds variances. We considered every time the worst case, where the Thymio only goes on the x-axis (for $Q_x$) or the y-axis (for $Q_y$). The time step $\Delta t=0.2s$. We have the following formulas : 
$$ Q_x = Q_y = \Delta t^2 Q_v = 0.017 mm^2 \\
Q_{\theta} = \Delta t^2 Q_w =  9.6 \cdot 10^{-4}rad^2$$



### R matrix - theoretical values

For the R matrix we had two different approach as the measurements came fromt two differents sources. 
#### $R_x, R_y, R_{\theta}$
We designed an experiment where we placed the Thymio on the side of the map, where the distorsion is maximal, and we monitored the measured position. From that we calculated the variance of the measurements. We found the following results :
$$ R_x = R_y = 0.68 mm^2 \\
R_{\theta} = 0.019 rad^2$$

#### $R_v, R_w$
We calculated the linear speed of each motor by running the Thymio at nominal speed (100 Thymio units) for 20s and sampling the speed value every 0.2s. We got the following results :

<img src="./Images/variance_rvr_rvl.png"  style="display: block; margin-left: auto; margin-right: auto; width: 50%; height: auto;">

We got the following results after conversion : 
$$R_{v}=2.83 mm^2/s^2 \\
R_{w}= 0.00125 rad^2/s^2$$

### Tuning
Unfortunately these parameters were not conclusive as we had some inconsistencies. The main problem came from the $Q_x, Q_y, Q_{\theta}$ terms as they are smaller than the $R_x, R_y, R_{\theta}$ terms. This led to the situation where the Kalman filter trusted more the prediction step than the measurements. We took inspiration from the advices of Prof. Ferrari Trecate in his course Multivariable Control where he said that we shouldn't hesitate to increase Q matrix values to have a working Kalman filter. Our hypothesis is that the process variances we found didn't take into account some effects that are not modeled in the system but still have an impart (wheel slippage for example).
After some tuning we found that these following values worked : 

$ Q_x = Q_y = Q_{xprev}\cdot 10^{2}= 1.7 mm^2 \\
Q_{\theta} = Q_{\theta prev}\cdot 10^{2}=0.096 rad^2 \\
Q_v = 0.43 mm^2/s^2 \\
Q_w = 0.0125 rad^2/s^2 \\
R_x = R_y = 0.68 mm^2 \\
R_{\theta} = 0.019 rad^2 \\
R_{v}=R_{vprev}\cdot 10^{-1}=0.283 mm^2/s^2 \\
R_{w}= 0.00125 rad^2/s^2$


## Global navigation

To navigate according to the path planned with A* algorithm, the control algorithm has to perform an error computation between the robot pose and the next node to reach. We used the Astolfi controller, as proposed in the course of week 1. It takes into error three different "errors":
- the euclidean distance between the goal and the state.
- the error in angle between the angle of the robot ($\theta$) and the angle the robot should have to go towards the node ($\theta$<sub>dir</sub>), denoted as $\alpha$
- the error in angle between the angle of the robot ($\theta$) and the optimal angle when reaching the node, decided by path planning ($\theta$<sub>g</sub>) , denoted as $\beta$

$\alpha = \theta$<sub>dir</sub>$ - \theta$

$\beta = \theta$<sub>g</sub>$ - \theta - \alpha$

$v = K$<sub>DIST</sub>$*\rho$

$w = K$<sub>ALPHA</sub>$*\alpha + K$<sub>BETA</sub>$*\beta$

<figure>
<img src="./Images/astolfi_control.png" alt="Geometric parameters in Astolfi control law" style="display: block; margin-left: auto; margin-right: auto; width: 50%; height: auto;">
<figcaption style="text-align: center;">Geometric parameters in Astolfi control law (slides week 1, "Components of a Mobile Robot", Prof. Mondada) </figcaption>
</figure>
  
The error in distance will influence the linear speed, while errors in angle will affect the angular, differential speed of the Thymio. The Astolfi control law is an exponentially stable law (i.e. guaranteed to converge towards the desired position with desired angle), under some conditions on gains K<sub>DIST</sub>, K<sub>ALPHA</sub> and K<sub>BETA</sub>.

$K$<sub>DIST</sub>$ > 0$

$K$<sub>DIST</sub>$ < 0$

$K$<sub>ALPHA</sub>$ - K$<sub>DIST</sub>$ > 0$




## Local avoidance

### Obstacle avoidance
In this scnenario, an object placed on the map would prevent the Thymio from following the optimal planned path without craching with it. 

In order to avoid the collision, the Thymio relies on the obstacle_detected() function. It checks if one of the horizontal proximity sensors has a abnormally large value .The threshold has been set to 20, as the sensors have a non-linear behavior, switching from reading 0 to 1000 at approximately 15 cm for a white and flat obstacle. The sensors readings then increases linearly as the distance decreases. Once an obstacle has been detected, the control() function is no longer in charge of the control and the local_avoidance() function takes control. It uses a single layer Artificial Neural Network to assign a speed to left and right motors. This is a linear combination of weights and readings of horizontal proximity sensors placed at the front and back of the Thymio robot. This function is based on the ANN implementation provided in the solution of week 3 exercise session. Weights have been adapted to give a satisfactory behavior.

<figure>
<img src="./Images/ANN_robot_control.png" alt="ANN scheme for obstacle avoidance" style="display: block; margin-left: auto; margin-right: auto; width: 50%; height: auto;">
<figcaption style="text-align: center;">ANN scheme for obstacle avoidance </figcaption>
</figure>

These two functions are integrated in the finite state machine of the main loop. The obstacle_detected() function is used as a transition to and out of the mode OBSTACLE_DETECTED, and local_avoidance() is the code exectued while in this mode.

When executing this mode (i.e. no more obstacle detected by obstacle_detected(), returning False), a new optimal path is recomputed, as the obstacle might have modified which path is the optimal path. The Thymio is first ordered to blindly move forward during a bit of time to clear the obstacle, and then to stop during optimal path computation.

One downside of our implementation with "flat" global obstacles and 3D local obstacles is that robot could mistakenly drive on a gloabl obstacle when in OBSTACLE_AVOIDANCE; we cannot handle all local avoidance scenarios and have to place our local obstacles on "program-friendly" locations. Ideally, the obstacle should not deviate the robot towards a global ostacle.

### Kidnapping scenarios
In this other scenario, the Thymio robot would be "kidnapped " and moved of a large distance, losing all its knowledges on its position and path to follow.

To detect the kidnapping, we use the proximity sensors located behind, reflecting the ground. The reading very quickly goes to zero as the robot is lifted. We check this in the boolean kidnapping_detected() function, testing whether the proximity.ground.reflected readings are under a predetermined threshold or not. This function acts as a transition towards and out of the mode KIDNAPPED. When in this mode, the only task of the program is to stop both motors.

Once the Thymio is placed on the ground, a new optimal path is computed. As the operator of the Thymio could unintentionnally mask the AruCo of the robot when putting the robot, a 3 seconds delay is given to correctly place the robot, before the camera takes a picture of the configuration.

It should be noted that this is the most restricting mode of the program. Since some confusion can occur when lifting the robot (maybe the operator masks some horizontal proximity sensors, wrongly entering OBSTACLE_AVOIDANCE mode), it is a prioritary mode. As such, the operator should take care to kidnap the robot without masking the proximity.ground sensors, as it would exit the kidnapping scenario !


## Live plots 
# Video

<video controls width="600">
  <source src="Images/video.mp4" type="video/mp4">
  Your browser does not support the video tag.
</video>


https://drive.google.com/file/d/1DII2xBjSa2V9xIoiqMG_Qr4aEIAdnn8P/view?usp=drive_link 



## Conclusion
To sum up, this project helped us discover new things and apply concepts we learned in class. As expected, we saw that the hardest part of the project was not to have different working modules, but a complete functionning project. Indeed, even though we managed pretty well to create our functions separately, we had quite a hard time putting it all together. Apart from that, the time we spent tuning the system was surprisingly low. In the end, we are satisfied with the result as everything worked nicely. 


In [1]:
"""
File name: main.ipynb
Authors: Jin Kilidjian, Corentin Plumet, Jules Bervillé, Charles Proffit 
Created: 05/12/24
Description: main loop of the program
"""

## Public libraries
import time
import numpy as np
import matplotlib.pyplot as plt 
import cv2
import cv2.aruco as aruco
%matplotlib qt

## Local custom libraries
from vision import initialize_vision,get_robot_position,get_initialization
import parameters as params
import global_path as gp
import local_navigation as locnav
import ekf_5 as kf

## Tdmclient asynchrone connexion
from tdmclient import ClientAsync, aw
client = ClientAsync()
node = await client.wait_for_node()
await node.lock()

OpenCV version: 4.10.0
Aruco available: True


Node e092c500-aaf6-40ff-88df-0dd608e53f2c

In [2]:
def motors(left, right):
    return {
        "motor.left.target": [left],
        "motor.right.target": [right],
    }

def set_motors(node,v_left,v_right):
    aw(node.set_variables(motors(int(v_left),int(v_right))))

def stop_motors(node):
    aw(node.set_variables(motors(0,0)))

cap = cv2.VideoCapture(0)

In [ ]:
# Initialize variables
mode = params.FOLLOW_PATH
mode_camera = params.NOMINAL
states = [] # x[mm],y[mm],theta[rad],v[mm/s],w[rad/s]
Ps = [kf.R_cam_on]
dt = params.DT
timer_camera = 0

# Initialization of camera
while get_initialization():
    ret, initial_frame = cap.read()
    map, goal = initialize_vision(initial_frame) #map in cases, goal in mm
state = get_robot_position(initial_frame) #it is in [mm,mm,rad]
state.extend([0,0])
states.append(state)

# Initialization of path and live plot
padded_map = gp.padding(map)
path = gp.a_star_search(padded_map, states[-1][0:2], goal)
path = gp.sampling(path,n=params.SAMPLING)
path = gp.add_angle_to_path(path)
fig,line,ellipse = gp.simu(path,padded_map,states[-1])
path = gp.convert_to_mm(path) #path in mm x[mm],y[mm]
counter = 0
counter_max = len(path)

The destination cell is found
The Path is 
-> (147, 30) -> (146, 31) -> (145, 32) -> (145, 33) -> (144, 34) -> (144, 35) -> (143, 36) -> (143, 37) -> (142, 38) -> (142, 39) -> (141, 40) -> (141, 41) -> (141, 42) -> (140, 43) -> (140, 44) -> (139, 45) -> (139, 46) -> (138, 47) -> (138, 48) -> (137, 49) -> (137, 50) -> (136, 51) -> (136, 52) -> (136, 53) -> (135, 54) -> (134, 55) -> (133, 56) -> (132, 57) -> (131, 58) -> (130, 59) -> (129, 60) -> (128, 61) -> (127, 62) -> (126, 63) -> (125, 64) -> (124, 65) -> (123, 66) -> (122, 67) -> (121, 68) -> (120, 69) -> (119, 70) -> (118, 71) -> (117, 72) -> (116, 73) -> (115, 74) -> (114, 75) -> (113, 76) -> (112, 77) -> (111, 78) -> (110, 79) -> (109, 80) -> (108, 81) -> (107, 82) -> (107, 83) -> (107, 84) -> (106, 85) -> (106, 86) -> (106, 87) -> (106, 88) -> (105, 89) -> (105, 90) -> (105, 91) -> (104, 92) -> (104, 93) -> (104, 94) -> (104, 95) -> (103, 96) -> (103, 97) -> (103, 98) -> (102, 99) -> (102, 100) -> (102, 101) -> (101, 102) -> (1

In [4]:
aw(node.set_variables({"button.center":[0]}))

while True:
     t_start = time.time()

     ## WAITING FOR NEW CAMERA VALUES
     timer_camera += dt
     if (timer_camera < params.T_UPDATE_CAM):
          z = [-1,-1,-1]
     else :
          ret, frame = cap.read()
          z = get_robot_position(frame) #in mm,mm,rad
          if z[0] < 0 and z[1] < 0 :
               if mode_camera == params.NOMINAL:
                    mode_camera = params.OBSTRUCTED
                    print("Camera obstructed")
          else :
               if mode_camera == params.OBSTRUCTED:
                    mode_camera = params.NOMINAL
                    print("Camera working")
               timer_camera = 0
     

     ## WAITING FOR NEW THYMIO VALUES ##
     aw(node.wait_for_variables({"motor.left.speed","motor.right.speed","button.center","prox.horizontal","prox.ground.reflected"}))
     speed_left = node["motor.left.speed"]
     speed_right = node["motor.right.speed"]
     button_center = node["button.center"]
     prox = node["prox.horizontal"]
     prox_ground = node["prox.ground.reflected"]

     ## KALMAN FILTER ##
     if (mode == params.FOLLOW_PATH) or (mode == params.OBSTACLE_AVOIDANCE):
          v,w = locnav.get_odometry(speed_right,speed_left)
          z.extend([v,w])
          state,P = kf.estimate_position(states[-1], z ,Ps[-1], dt)
          states.append(state)
          Ps.append(P)
          var_x = Ps[-1][0][0]
          var_y = Ps[-1][1][1]    
          locnav.update_plots(states,Ps[-1],fig,line,ellipse)


     ## FINITE STATE MACHINE ##
     if (button_center==1):
          print("Button pressed")
          break

     if locnav.goal_is_reached(states,path) :
          print("Arrived at goal")
          break

     if locnav.node_is_reached(states,path,counter) :
          print("Arrived at node",counter)
          counter += 1

     if  mode == params.FOLLOW_PATH :
          if locnav.kidnapping_detected(prox_ground):
               print("Kidnapped")
               mode = params.KIDNAPPED
          elif locnav.object_detected(prox):
               print("Obstacle detected")
               mode = params.OBSTACLE_AVOIDANCE
          else: # goal tracking: turn toward the goal
               v_left,v_right = locnav.control(states[-1],path[counter],(counter - counter_max) == 0)
               set_motors(node,v_left,v_right)
     if mode == params.OBSTACLE_AVOIDANCE:
          if locnav.kidnapping_detected(prox_ground):
               print("Kidnapped")
               mode = params.KIDNAPPED
          elif locnav.object_detected(prox): # obstacle avoidance: accelerate wheel near obstacle
               v_left,v_right = locnav.local_avoidance(prox)
               set_motors(node,v_left,v_right)
          else:
               set_motors(node,params.NOMINAL_SPD,params.NOMINAL_SPD)
               aw(client.sleep(3))
               stop_motors(node)
               print("Obstacle passed")
               aw(client.sleep(3))
               # Reinitialization of path and live plot
               mode = params.FOLLOW_PATH
               ret, initial_frame = cap.read()
               state = get_robot_position(initial_frame)
               state.extend([0,0])
               states = []
               states.append(state)
               Ps.append(kf.R_cam_on)
               mode = params.FOLLOW_PATH
               path = gp.a_star_search(padded_map, states[-1][0:2], goal)
               path = gp.sampling(path,n=params.SAMPLING)
               path = gp.add_angle_to_path(path)
               fig,line,ellipse = gp.simu(path,padded_map,states[-1])
               path = gp.convert_to_mm(path) #path in mm x[mm],y[mm]
               counter = 0
               counter_max = len(path)
     if mode == params.KIDNAPPED:
          if locnav.kidnapping_detected(prox_ground):
               stop_motors(node)
          else:
               print("Released")
               aw(client.sleep(3))
               # Reinitialization of path and live plot
               ret, initial_frame = cap.read()
               state = get_robot_position(initial_frame)
               state.extend([0,0])
               states = []
               states.append(state)
               Ps.append(kf.R_cam_on)
               mode = params.FOLLOW_PATH
               path = gp.a_star_search(padded_map, states[-1][0:2], goal)
               path = gp.sampling(path,n=params.SAMPLING)
               path = gp.add_angle_to_path(path)
               fig,line,ellipse = gp.simu(path,padded_map,states[-1])
               path = gp.convert_to_mm(path) #path in mm x[mm],y[mm]
               counter = 0
               counter_max = len(path)
     
     ## TIMING THE LOOP TO LAST DT ##
     dt = time.time() - t_start
     if dt < params.DT:
          dt = params.DT
          aw(client.sleep(params.DT-dt))

stop_motors(node) 

Arrived at node 0
Arrived at node 1
Arrived at node 2
Arrived at node 3
Arrived at node 4
Arrived at node 5
Obstacle detected
Arrived at node 6
Obstacle passed
The destination cell is found
The Path is 
-> (98, 185) -> (97, 186) -> (96, 187) -> (95, 188) -> (94, 189) -> (93, 190) -> (92, 191) -> (91, 192) -> (90, 193) -> (89, 194) -> (88, 195) -> (87, 196) -> (86, 197) -> (85, 198) -> (84, 199) -> (83, 200) -> (82, 201) -> (81, 202) -> (80, 203) -> (79, 204) -> (78, 205) -> (77, 206) -> (76, 207) -> (75, 208) -> (74, 209) -> (73, 210) -> (72, 211) -> (71, 212) -> (70, 213) -> (69, 214) -> (68, 215) -> (67, 216) -> (66, 217) -> (65, 218) -> (64, 219) -> (63, 220) -> (62, 221) -> (61, 222) -> (60, 223) -> (59, 224) -> (58, 225) -> (57, 226) -> (56, 227) -> (56, 228) -> (55, 229) -> (55, 230) -> (55, 231) -> (55, 232) -> (54, 233) -> (53, 234) -> (53, 235) -> (52, 236) -> (52, 237) -> (51, 238) -> (51, 239) -> (50, 240) -> (50, 241) -> (50, 242) -> (49, 243) -> (49, 244) -> (48, 245) -> (